In [ ]:
import os

BASE_URL = os.getenv(
    "API_URL",
    "https://alphasignal-dev.moretoncp.com"
)

TOKEN = os.getenv("API_TOKEN")

# Later you set:
# export API_TOKEN="your_jwt_here"

In [1]:
import time
import json
from api_client import TradingDeskAPI
from database import save_snapshots

counter = 0

def should_collect(market):

    prices = json.loads(market["outcomePrices"])
    yes_price = float(prices[0])

    if yes_price < 0.05 or yes_price > 0.95:
        return False

    return True

api = TradingDeskAPI()

def collect(counter):

    markets = api.get_markets()
    print(len(markets))

    snapshots = []

    for market in markets:

        print(market)
        print(market["outcomePrices"])

        if not market["acceptingOrders"]:
            continue

        if market.get("ended"):
            continue

        if market.get("closed"):
            continue

        if not market.get("enableOrderBook"):
            continue

        token_ids = json.loads(market["clobTokenIds"])

        if not token_ids:
            print('no token ids')
            continue

        yes_token = token_ids[0]

        book = api.get_orderbook(yes_token)

        if not book.get("bids") or not book.get("asks"):
            print("empty orderbook")
            continue

        best_bid = float(book["bids"][0]["price"])
        best_ask = float(book["asks"][0]["price"])

        spread = best_ask - best_bid

        # if spread > 0.03:
        #     print('wide spread')
        #     continue

        snapshot = {
            "condition_id": market["conditionId"],
            "question": market["question"],
            "description": market["description"],
            "volume": market["volume"],
            "volumeNum": market["volumeNum"],
            "liquidityNum": market["liquidityNum"],
            "orderPriceMinTickSize": market["orderPriceMinTickSize"],
            "orderMinSize": market["orderMinSize"],
            "best_bid": best_bid,
            "best_ask": best_ask,
            "spread": spread,
            "orderbook": json.dumps(book),
            "events": json.dumps(market["events"]),
            "taker_fee_rate": market["feeSchedule"]["rate"] if market["feesEnabled"] == True else 0,
            "yes_price": float(json.loads(market["outcomePrices"])[0]),
            "timestamp": time.time(),
            "token_id": yes_token,
        }

        snapshots.append(snapshot)

        print(
            snapshot["question"],
            snapshot["yes_price"]
        )

        # break
    save_snapshots(snapshots, counter)

try:
    collect(counter)

except Exception as e:
    print("ERROR:", e)
# while True:

#     try:
#         collect(counter)

#     except Exception as e:
#         print("ERROR:", e)
#     counter += 1

#     time.sleep(300)

100
{'id': '1105740', 'question': 'Will Edmundo González be the leader of Venezuela end of 2026?', 'conditionId': '0xb122b2a17c8dea01d5e8ffe04316bbe011f9bf80c989bb82afde2f39a3fd4441', 'slug': 'will-edmundo-gonzlez-be-the-leader-of-venezuela-end-of-2026', 'resolutionSource': '', 'endDate': '2026-12-31T00:00:00Z', 'liquidity': '99790.83912', 'startDate': '2026-01-04T18:21:09.391Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/venezuela-leader-end-of-2026-lOfqbUxiKAsg.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/venezuela-leader-end-of-2026-lOfqbUxiKAsg.png', 'description': 'This market will resolve to the individual who officially holds the position of the head of state of Venezuela on Dec 31, 2026 at 12 PM ET.\n\nFor the purposes of this market, "officially holds" refers to the individual that was formally appointed, confirmed (if confirmation is required), and sworn in as the head of state of Venezuela or otherwise confirmed by official governme

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4